In [ ]:
#Jacobi and Gauss-Seidal method of solving linear equations
import numpy as np

In [ ]:
def reorder_for_nonzero_diagonal(A, b):
  n = len(b)
  A = A.copy()
  b = b.copy()
  swaps = []

  for i in range(n):
    if A[i, i] != 0:
      continue

    found = None
    for j in range(n):
      if j != i and A[j, i] != 0 and A[i, j] != 0:
        found = j
        break
      if found is None:
        for j in range(n):
          if j != i and A[j, i] != 0:
            found = j
            break
    if found is None:
      raise ValueError(f"No suitable row to swap with. The system may be singular.")
    A[[i, found]] = A[[found, i]]
    b[i], b[found] = b[found], b[i]
    swaps.append((i + 1, found + 1))

  return A, b, swaps

def is_diagonally_dominant(A, n):
    for i in range(n):
        diag = abs(A[i, i])
        off_sum = sum(abs(A[i, j]) for j in range(n) if j != i)
        if diag <= off_sum:
            return False
    return True

def input_matrix(n):
  #Co-effecient matrix A, size n x n
  A = []
  for i in range(n):
    while True:
      try:
        row = list(map(float, input(f"Enter {n} coefficients for row {i+1} (space-separated): ").split()))
        if len(row) != n:
          print(f"ERROR: Expected {n} values.")
          continue
        A.append(row)
        break
      except ValueError:
        print("ERROR: Invalid input. Please enter valid inputs.")
  return np.array(A, dtype=float)

def input_vector(n, label):
  v = []
  for i in range(n):
    while True:
      try:
        v.append(float(input(f"Enter {n} {label} (space-separated): ")))
        break
      except ValueError:
        print("ERROR: Invalid input. Please enter valid inputs.")
  return np.array(v, dtype=float)

def jacobi(A, b, x0, tol, max_iter):
  n = len(b)
  x = x0.copy()
  info = []

  for k in range(1, max_iter + 1):
    x_new = np.zeros(n)
    for i in range(n):
      sigma = sum(A[i, j] * x[j] for j in range(n) if j != i)
      x_new[i] = (b[i] - sigma) / A[i, i]

    abs_errors = np.abs(x_new - x)
    residual = np.linalg.norm(np.dot(A, x_new) - b)
    info.append((k, x_new.copy(), abs_errors.copy(), residual))

    if np.max(abs_errors) < tol:
      return x_new, info, True
    x = x_new
  return x, info, False #solution vector, info, convergence status

def gauss_seidel(A, b, x0, tol, max_iter):
  n = len(b)
  x = x0.copy()
  info = []

  for k in range(1, max_iter + 1):
    x_new = x.copy()
    for i in range(n):
      sigma = sum(A[i, j] * x_new[j] for j in range(n) if j != i)
      x_new[i] = (b[i] - sigma) / A[i, i]

    abs_errors = np.abs(x_new - x)
    residual = np.linalg.norm(A @ x_new - b)
    info.append((k, x_new.copy(), abs_errors.copy(), residual))

    if np.max(abs_errors) < tol:
      return x_new, info, True
    x = x_new

  return x, info, False #solution vector, info, convergence status

def print_separator(char="─", width=70):
  print(char * width)

def print_header(title):
  print_separator("═")
  print(f"  {title}")
  print_separator("═")

def print_iteration_table(history, n):
  # Column headers
  var_headers = "  ".join(f"{'x'+str(i+1):>12}" for i in range(n))
  err_headers = "  ".join(f"{'|Δx'+str(i+1)+'|':>12}" for i in range(n))
  print(f"\n{'Iter':>4}  {var_headers}  {err_headers}  {'Residual':>14}")
  print_separator()

  for (k, x, errs, res) in history:
    x_str   = "  ".join(f"{xi:>12.6f}" for xi in x)
    err_str = "  ".join(f"{e:>12.6f}" for e in errs)
    print(f"{k:>4}  {x_str}  {err_str}  {res:>14.6e}")

def print_results(method_name, x, history, converged, n):
  print_header(f"{method_name} Results")

  print_iteration_table(history, n)
  print_separator()

  if converged:
    k_final = history[-1][0]
    print(f"\nConverged in {k_final} iteration(s).")
  else:
    print(f"\nDid NOT converge within {len(history)} iterations.")

  print("\n  Solution vector:")
  for i, xi in enumerate(x):
    print(f"    x{i+1} = {xi:.8f}")

  final_res = history[-1][3]
  print(f"\n  Final residual ‖Ax − b‖₂ = {final_res:.6e}")

def print_comparison(jac_hist, gs_hist, jac_conv, gs_conv):
  print_header("Comparative Analysis")

  j_iters = len(jac_hist)
  g_iters = len(gs_hist)

  print(f"  {'Method':<20} {'Iterations':>12} {'Final Residual':>18} {'Converged':>12}")
  print_separator("-", 70)
  print(f"  {'Jacobi':<20} {j_iters:>12} {jac_hist[-1][3]:>18.6e} {'Yes' if jac_conv else 'No':>12}")
  print(f"  {'Gauss-Seidel':<20} {g_iters:>12} {gs_hist[-1][3]:>18.6e} {'Yes' if gs_conv else 'No':>12}")

  if jac_conv and gs_conv:
    faster = "Gauss-Seidel" if g_iters < j_iters else ("Jacobi" if j_iters < g_iters else "Both (tied)")
    ratio = j_iters / g_iters if g_iters > 0 else float("inf")
    print(f"\n  Faster method : {faster}")
    print(f"  Speed ratio   : Jacobi took {ratio:.2f}× the iterations of Gauss-Seidel")
    print("\n  Remark: Gauss-Seidel typically converges roughly twice as fast as")
    print("  Jacobi because it uses updated values within the same iteration.")

In [ ]:
#main
n = int(input("Enter number of equations (n ≥ 2): "))

A = input_matrix(n)
b = input_vector(n, "constants (b)") #constant values b
x0 = input_vector(n, "init_guess (x0)") #initial guesses x0
max_iter = int(input("Input number of iterations: "))
toler = float(input("Input convergence tolerance, e: "))

for i in range(n):
  if A[i, i] == 0:
    print(f"A[{i+1}][{i+1}] = 0  → will be fixed by row swap")
A, b, swaps = reorder_for_nonzero_diagonal(A, b)

print()
if (is_diagonally_dominant(A, n)):
  print("The matrix is diagonally dominant. Hence convergence is guranteed.")
else:
  print("The matrix is not diagonally dominant. Hence convergence is not guranteed.")

x_jac, jac_hist, jac_conv = jacobi(A, b, x0, toler, max_iter)
x_gs,  gs_hist,  gs_conv  = gauss_seidel(A, b, x0, toler, max_iter)

print_results("Jacobi Method",       x_jac, jac_hist, jac_conv, n)
print_results("Gauss-Seidel Method", x_gs,  gs_hist,  gs_conv,  n)
print_comparison(jac_hist, gs_hist, jac_conv, gs_conv)

Enter number of equations (n ≥ 2): 3
Enter 3 coefficients for row 1 (space-separated): 4 -1 0
Enter 3 coefficients for row 2 (space-separated): -1 4 -1
Enter 3 coefficients for row 3 (space-separated): 0 -1 4
Enter 3 constants (b) (space-separated): 12
Enter 3 constants (b) (space-separated): 1
Enter 3 constants (b) (space-separated): 5
Enter 3 init_guess (x0) (space-separated): 0
Enter 3 init_guess (x0) (space-separated): 0
Enter 3 init_guess (x0) (space-separated): 0
Input number of iterations: 50
Input convergence tolerance, e: 0.0001

The matrix is diagonally dominant. Hence convergence is guranteed.
══════════════════════════════════════════════════════════════════════
  Jacobi Method Results
══════════════════════════════════════════════════════════════════════

Iter            x1            x2            x3         |Δx1|         |Δx2|         |Δx3|        Residual
──────────────────────────────────────────────────────────────────────
   1      3.000000      0.250000      1.25000

In [ ]:
#main
n = int(input("Enter number of equations (n ≥ 2): "))

A = input_matrix(n)
b = input_vector(n, "constants (b)") #constant values b
x0 = input_vector(n, "init_guess (x0)") #initial guesses x0
max_iter = int(input("Input number of iterations: "))
toler = float(input("Input convergence tolerance, e: "))

for i in range(n):
  if A[i, i] == 0:
    print(f"A[{i+1}][{i+1}] = 0  → will be fixed by row swap")
A, b, swaps = reorder_for_nonzero_diagonal(A, b)

print()
if (is_diagonally_dominant(A, n)):
  print("The matrix is diagonally dominant. Hence convergence is guranteed.")
else:
  print("The matrix is not diagonally dominant. Hence convergence is not guranteed.")

x_jac, jac_hist, jac_conv = jacobi(A, b, x0, toler, max_iter)
x_gs,  gs_hist,  gs_conv  = gauss_seidel(A, b, x0, toler, max_iter)

print_results("Jacobi Method",       x_jac, jac_hist, jac_conv, n)
print_results("Gauss-Seidel Method", x_gs,  gs_hist,  gs_conv,  n)
print_comparison(jac_hist, gs_hist, jac_conv, gs_conv)

Enter number of equations (n ≥ 2): 3
Enter 3 coefficients for row 1 (space-separated): 0 -1 4
Enter 3 coefficients for row 2 (space-separated): -1 4 -1
Enter 3 coefficients for row 3 (space-separated): 4 -1 0
Enter 3 constants (b) (space-separated): 12
Enter 3 constants (b) (space-separated): 1
Enter 3 constants (b) (space-separated): 5
Enter 3 init_guess (x0) (space-separated): 0
Enter 3 init_guess (x0) (space-separated): 0
Enter 3 init_guess (x0) (space-separated): 0
Input number of iterations: 50
Input convergence tolerance, e: 0.0001
A[1][1] = 0  → will be fixed by row swap
A[3][3] = 0  → will be fixed by row swap

The matrix is not diagonally dominant. Hence convergence is not guranteed.
══════════════════════════════════════════════════════════════════════
  Jacobi Method Results
══════════════════════════════════════════════════════════════════════

Iter            x1            x2            x3         |Δx1|         |Δx2|         |Δx3|        Residual
──────────────────────────

In [ ]:
#main
n = int(input("Enter number of equations (n ≥ 2): "))

A = input_matrix(n)
b = input_vector(n, "constants (b)") #constant values b
x0 = input_vector(n, "init_guess (x0)") #initial guesses x0
max_iter = int(input("Input number of iterations: "))
toler = float(input("Input convergence tolerance, e: "))

for i in range(n):
  if A[i, i] == 0:
    print(f"A[{i+1}][{i+1}] = 0  → will be fixed by row swap")
A, b, swaps = reorder_for_nonzero_diagonal(A, b)

print()
if (is_diagonally_dominant(A, n)):
  print("The matrix is diagonally dominant. Hence convergence is guranteed.")
else:
  print("The matrix is not diagonally dominant. Hence convergence is not guranteed.")

x_jac, jac_hist, jac_conv = jacobi(A, b, x0, toler, max_iter)
x_gs,  gs_hist,  gs_conv  = gauss_seidel(A, b, x0, toler, max_iter)

print_results("Jacobi Method",       x_jac, jac_hist, jac_conv, n)
print_results("Gauss-Seidel Method", x_gs,  gs_hist,  gs_conv,  n)
print_comparison(jac_hist, gs_hist, jac_conv, gs_conv)

Enter number of equations (n ≥ 2): 3
Enter 3 coefficients for row 1 (space-separated): 1 2 -2
Enter 3 coefficients for row 2 (space-separated): 1 1 1
Enter 3 coefficients for row 3 (space-separated): 2 -1 0
Enter 3 constants (b) (space-separated): 2
Enter 3 constants (b) (space-separated): 6
Enter 3 constants (b) (space-separated): 3
Enter 3 init_guess (x0) (space-separated): 0
Enter 3 init_guess (x0) (space-separated): 0
Enter 3 init_guess (x0) (space-separated): 0
Input number of iterations: 50
Input convergence tolerance, e: 0.0001
A[3][3] = 0  → will be fixed by row swap

The matrix is not diagonally dominant. Hence convergence is not guranteed.
══════════════════════════════════════════════════════════════════════
  Jacobi Method Results
══════════════════════════════════════════════════════════════════════

Iter            x1            x2            x3         |Δx1|         |Δx2|         |Δx3|        Residual
──────────────────────────────────────────────────────────────────────